<a href="https://colab.research.google.com/github/healan/JobLens/blob/main/jobMatching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install & import necessary package

In [ ]:
"""

2. Baseline: TF-IDF consine similarity
3. Proposed: Sentence-BERT (all-MiniLM-L6-v2) consine similarity
4. evaluation: Precision@K, NDCG, Qualitative Comparison
5. save result matrix/graph
"""

'\n1. Load resume-job description pair data\n2. Baseline: TF-IDF consine similarity\n3. Proposed: Sentence-BERT (all-MiniLM-L6-v2) consine similarity\n4. evaluation: Precision@K, NDCG, Qualitative Comparison\n5. save result matrix/graph\n'

In [ ]:
#!pip install sentence-transformers pandas scikit-learn -q
#!pip install datasets pandas -q
#!pip install huggingface_hub -q
## !pip install kagglehub[pandas-datasets]

In [ ]:
import random
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import HfApi
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation
)
from torch.utils.data import DataLoader
random.seed(42)
np.random.seed(42)

/tmp/ipykernel_2000/3302907335.py:9: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_2000/3302907335.py:9: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import (


### Load Resume data (hugging face)
https://huggingface.co/datasets/ahmedheakl/resume-atlas


In [ ]:
from datasets import load_dataset

hf_dataset = load_dataset("ahmedheakl/resume-atlas")
resumes_raw = hf_dataset["train"].to_pandas()

resumes_df = pd.DataFrame({
    "id": [f"R{i}" for i in range(len(resumes_raw))],
    "category": resumes_raw["Category"].str.strip(),
    "resume_text": resumes_raw["Text"]
})


README.md:   0%|          | 0.00/215 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/53.6M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

### EDA

In [ ]:
resumes_df.category.unique()

array(['Accountant', 'Advocate', 'Agriculture', 'Apparel', 'Architecture',
       'Arts', 'Automobile', 'Aviation', 'Banking', 'Blockchain', 'BPO',
       'Building and Construction', 'Business Analyst', 'Civil Engineer',
       'Consultant', 'Data Science', 'Database', 'Designing', 'DevOps',
       'Digital Media', 'DotNet Developer', 'Education',
       'Electrical Engineering', 'ETL Developer', 'Finance',
       'Food and Beverages', 'Health and Fitness', 'Human Resources',
       'Information Technology', 'Java Developer', 'Management',
       'Mechanical Engineer', 'Network Security Engineer',
       'Operations Manager', 'PMO', 'Public Relations',
       'Python Developer', 'React Developer', 'Sales', 'SAP Developer',
       'SQL Developer', 'Testing', 'Web Designing'], dtype=object)

In [ ]:
resumes_df.head()

,id,category,resume_text
0,R0,Accountant,education omba executive leadership university...
1,R1,Accountant,howard gerrard accountant deyjobcom birmingham...
2,R2,Accountant,kevin frank senior accountant inforesumekraftc...
3,R3,Accountant,place birth nationality olivia ogilvy accounta...
4,R4,Accountant,stephen greet cpa senior accountant 9 year exp...


In [ ]:
resumes_df.category.unique()

array(['Accountant', 'Advocate', 'Agriculture', 'Apparel', 'Architecture',
       'Arts', 'Automobile', 'Aviation', 'Banking', 'Blockchain', 'BPO',
       'Building and Construction', 'Business Analyst', 'Civil Engineer',
       'Consultant', 'Data Science', 'Database', 'Designing', 'DevOps',
       'Digital Media', 'DotNet Developer', 'Education',
       'Electrical Engineering', 'ETL Developer', 'Finance',
       'Food and Beverages', 'Health and Fitness', 'Human Resources',
       'Information Technology', 'Java Developer', 'Management',
       'Mechanical Engineer', 'Network Security Engineer',
       'Operations Manager', 'PMO', 'Public Relations',
       'Python Developer', 'React Developer', 'Sales', 'SAP Developer',
       'SQL Developer', 'Testing', 'Web Designing'], dtype=object)

### Load job posting data
https://www.kaggle.com/datasets/arshkon/linkedin-job-postings

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "postings.csv"

jobs_raw = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "arshkon/linkedin-job-postings",
    file_path,
)


print("column name:", jobs_raw.columns.tolist())
print(f"\nTotal {len(jobs_raw)} row")

100%|██████████| 147M/147M [00:07<00:00, 20.7MB/s]

Extracting zip of postings.csv...


column name: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']

Total 123849 row


In [ ]:
jobs_df = pd.DataFrame({
    "id": [f"J{i}" for i in range(len(jobs_raw))],
    "title": jobs_raw["title"],
    "job_description": jobs_raw["description"]
})

In [ ]:
jobs_df.head()

,id,title,job_description
0,J0,Marketing Coordinator,Job descriptionA leading real estate firm in N...
1,J1,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ..."
2,J2,Assitant Restaurant Manager,The National Exemplar is accepting application...
3,J3,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...
4,J4,Service Technician,Looking for HVAC service tech with experience ...


In [ ]:
# (ground truth용 proxy label)
# left side(key) → resume-atlas real category name(perfect match)
# right side(value) → search in Linkedin job posting title (can include few charactor)

job_category_map = {
    "Accountant":                   "accountant",
    "Advocate":                     "advocate",
    "Agriculture":                  "agricultur",
    "Apparel":                      "apparel",
    "Architecture":                 "architect",
    "Arts":                         "arts",
    "Automobile":                   "automobil",
    "Aviation":                     "aviation",
    "Banking":                      "banking",
    "Blockchain":                   "blockchain",
    "BPO":                          "bpo",
    "Building and Construction":    "construction",
    "Business Analyst":             "business analyst",
    "Civil Engineer":               "civil engineer",
    "Consultant":                   "consultant",
    "Data Science":                 "data scien",
    "Database":                     "database",
    "Designing":                    "design",
    "DevOps":                       "devops",
    "Digital Media":                "digital media",
    "DotNet Developer":             "dotnet",
    "Education":                    "education",
    "Electrical Engineering":       "electrical",
    "ETL Developer":                "etl",
    "Finance":                      "financ",
    "Food and Beverages":           "food",
    "Health and Fitness":           "health",
    "Human Resources":              "human resource",
    "Information Technology":       "information technolog",
    "Java Developer":               "java",
    "Management":                   "management",
    "Mechanical Engineer":          "mechanical",
    "Network Security Engineer":    "network security",
    "Operations Manager":           "operations",
    "PMO":                          "pmo",
    "Public Relations":             "public relation",
    "Python Developer":             "python",
    "React Developer":              "react",
    "Sales":                        "sales",
    "SAP Developer":                "sap",
    "SQL Developer":                "sql",
    "Testing":                      "test",
    "Web Designing":                "web design",
}

In [ ]:
def find_category_for_job(title):
    for cat, keyword in job_category_map.items():
        if keyword.lower() in str(title).lower():
            return cat
    return None

jobs_df["matched_category"] = jobs_df["title"].apply(find_category_for_job)
jobs_df = jobs_df.dropna(subset=["matched_category"]).reset_index(drop=True)

In [ ]:
# --- 전체 resumes_df 기준으로 ground truth 매핑 먼저 생성 --- 각 카테고리 대표 이력서 1개를 정답으로 지정
np.random.seed(42)
category_to_resume = (
    resumes_df.groupby("category")["id"]
    .apply(lambda x: np.random.choice(x))
    .to_dict()
)

# {
#   "Data Science":   "R3",
#   "Java Developer": "R51",
#   "HR":             "R100",
#   ...
# }

# --- 공고 샘플링 ---
jobs_df = jobs_df.sample(n=min(50, len(jobs_df)), random_state=42).reset_index(drop=True)
jobs_df["correct_resume_id"] = jobs_df["matched_category"].map(category_to_resume)
jobs_df = jobs_df.dropna(subset=["correct_resume_id"]).reset_index(drop=True)

# 공고 제목               matched_category    correct_resume_id
# "Senior Data Scientist" → "Data Science"  → "R3"

print(f"\n success load data: Resume total : {len(resumes_df)}, posting total : {len(jobs_df)}")
print(f"   카테고리 분포: {resumes_df['category'].value_counts().to_dict()}\n")

# --- ground truth ID가 반드시 포함되도록 이력서 구성 ---
required_ids  = set(jobs_df["correct_resume_id"].unique())
required_rows = resumes_df[resumes_df["id"].isin(required_ids)]
remaining     = resumes_df[~resumes_df["id"].isin(required_ids)]
n_extra       = max(0, min(100, len(resumes_df)) - len(required_rows)) ## 500 -> 250 base line model evaluation was changed, but finetuned model evaluation is the same.
extra_rows    = remaining.sample(n=n_extra, random_state=42) if n_extra > 0 else pd.DataFrame()
resumes_df    = pd.concat([required_rows, extra_rows]).reset_index(drop=True)

print(f"\n✅ 데이터 로드 완료: 이력서 {len(resumes_df)}개, 공고 {len(jobs_df)}개")
print(f"   ground truth ID 포함 확인: {jobs_df['correct_resume_id'].isin(resumes_df['id']).all()}")
print(f"   카테고리 분포: {resumes_df['category'].value_counts().to_dict()}\n")


 success load data: Resume total : 250, posting total : 50
   카테고리 분포: {'Architecture': 13, 'SQL Developer': 12, 'Java Developer': 11, 'Business Analyst': 11, 'Automobile': 11, 'Apparel': 9, 'Electrical Engineering': 9, 'Consultant': 8, 'Finance': 8, 'Building and Construction': 8, 'Accountant': 8, 'Public Relations': 7, 'Designing': 7, 'Mechanical Engineer': 7, 'Civil Engineer': 7, 'Operations Manager': 6, 'DevOps': 6, 'Arts': 6, 'PMO': 6, 'React Developer': 6, 'DotNet Developer': 6, 'Agriculture': 6, 'ETL Developer': 6, 'Education': 6, 'Testing': 6, 'Human Resources': 5, 'Information Technology': 5, 'Banking': 5, 'Management': 4, 'Data Science': 4, 'BPO': 4, 'Health and Fitness': 3, 'SAP Developer': 3, 'Sales': 3, 'Aviation': 3, 'Network Security Engineer': 3, 'Digital Media': 3, 'Web Designing': 3, 'Food and Beverages': 2, 'Database': 2, 'Python Developer': 1, 'Advocate': 1}


✅ 데이터 로드 완료: 이력서 100개, 공고 50개
   ground truth ID 포함 확인: True
   카테고리 분포: {'Java Developer': 7, 'Informatio

In [ ]:
# ─────────────────────────────────────────────────────────
# STEP 3. evaluation function — Precision@K, NDCG
# ─────────────────────────────────────────────────────────
def evaluate(sim_matrix, jobs_df, resumes_df, k=3):
    """
    각 공고에 대해 상위 K개 이력서를 추천했을 때,
    정답 이력서가 그 안에 포함되는지(Precision@K),
    그리고 순위가 얼마나 좋은지(NDCG)를 계산
    """
    resume_ids = resumes_df["id"].tolist()
    hits = 0
    ndcg_scores = []

    detail_rows = []

    for i, row in jobs_df.iterrows():
        scores = sim_matrix[i]
        ranked_idx = np.argsort(scores)[::-1]  # 유사도 높은 순
        ranked_resume_ids = [resume_ids[idx] for idx in ranked_idx]
        top_k = ranked_resume_ids[:k]

        correct_id = row["correct_resume_id"]
        is_hit = correct_id in top_k
        hits += int(is_hit)

        # NDCG@K (단일 정답 기준 간소화 버전)
        if correct_id in ranked_resume_ids:
            rank = ranked_resume_ids.index(correct_id) + 1  # 1-indexed
            ndcg = 1 / np.log2(rank + 1) if rank <= k else 0
        else:
            ndcg = 0
        ndcg_scores.append(ndcg)

        detail_rows.append({
            "job_title": row["title"],
            "correct_resume": correct_id,
            "top1_predicted": ranked_resume_ids[0],
            "top1_score": round(scores[ranked_idx[0]], 3),
            "hit_at_k": is_hit
        })

    precision_at_k = hits / len(jobs_df)
    avg_ndcg = np.mean(ndcg_scores)

    return {
        "precision_at_k": round(precision_at_k, 3),
        "ndcg": round(avg_ndcg, 3),
        "details": pd.DataFrame(detail_rows)
    }

In [ ]:
# ─────────────────────────────────────────────────────────
# STEP 1. BASELINE — TF-IDF consine similarity
# ─────────────────────────────────────────────────────────
def run_tfidf_baseline(resumes, jobs):
    corpus = list(resumes["resume_text"]) + list(jobs["job_description"])
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(corpus)

    n_resumes = len(resumes)
    resume_vecs = tfidf_matrix[:n_resumes]
    job_vecs = tfidf_matrix[n_resumes:]

    sim_matrix = cosine_similarity(job_vecs, resume_vecs)  # shape: (n_jobs, n_resumes)
    return sim_matrix

In [ ]:
# ─────────────────────────────────────────────────────────
# STEP 2. Compare three  base models with out fine-tuning
# ─────────────────────────────────────────────────────────
def run_all_models(resumes, jobs):
    from sentence_transformers import SentenceTransformer

    models = {
        "all-MiniLM-L6-v2":   "all-MiniLM-L6-v2",  # Base Model, light & fast, 384 dimension
        "all-mpnet-base-v2":   "all-mpnet-base-v2", # high accurancy than mini, 768 dimension
        "BAAI/bge-base-en-v1.5": "BAAI/bge-base-en-v1.5", # IR specific model. new one
    }

    results = {}
    for name, model_id in models.items():
        print(f"\n🔄 {name} 실행 중...")
        model = SentenceTransformer(model_id)
        resume_embs = model.encode(resumes["resume_text"].tolist(), show_progress_bar=False)
        job_embs    = model.encode(jobs["job_description"].tolist(), show_progress_bar=False)
        sim_matrix  = cosine_similarity(job_embs, resume_embs)
        results[name] = sim_matrix
        print(f"✅ {name} 완료")
    return results

# 실행
all_sims = run_all_models(resumes_df, jobs_df)

# 평가
print("\n" + "="*60)
print("🎯 모델별 최종 비교")
print("="*60)

tfidf_sim    = run_tfidf_baseline(resumes_df, jobs_df)
tfidf_result = evaluate(tfidf_sim, jobs_df, resumes_df, k=3)

comparison_rows = [{"Model": "TF-IDF (Baseline)", #Matches resumes and job postings based on exact keyword overlap, without understanding context or meaning.
                    "Precision@3": tfidf_result["precision_at_k"], # Out of 3 recommended resumes, how many are actually correct.
                    "NDCG@3": tfidf_result["ndcg"]}] # Not just whether the correct resume appears in top 3, but how highly it is ranked.

for name, sim in all_sims.items():
    result = evaluate(sim, jobs_df, resumes_df, k=3)
    comparison_rows.append({
        "Model": name,
        "Precision@3": result["precision_at_k"],
        "NDCG@3": result["ndcg"]
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))
comparison_df.to_csv("model_comparison_all.csv", index=False)


🔄 all-MiniLM-L6-v2 실행 중...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

KeyboardInterrupt: 

## Finetuned Model

In [ ]:
# ─────────────────────────────────────────────────────────
# STEP 1. 학습 데이터 구성
# ─────────────────────────────────────────────────────────
# Fine-tuning 전략:
# Positive pair: (공고, 같은 카테고리 이력서) → label 1.0
# Negative pair: (공고, 다른 카테고리 이력서) → label 0.0
# 비율: Positive 1 : Negative 2 (불균형 방지)

print("🔄 학습 데이터 구성 중...")

train_examples = []

# 카테고리별 이력서 딕셔너리
cat_to_resumes = (
    resumes_df.groupby("category")["resume_text"]
    .apply(list)
    .to_dict()
)

for _, job in jobs_df.iterrows():
    job_text     = str(job["job_description"])[:512]
    job_category = job["matched_category"]

    # Positive: 같은 카테고리 이력서 2개
    pos_resumes = cat_to_resumes.get(job_category, [])
    if not pos_resumes:
        continue
    pos_samples = random.sample(pos_resumes, min(2, len(pos_resumes)))
    for pos in pos_samples:
        train_examples.append(InputExample(
            texts=[job_text, str(pos)[:512]],
            label=1.0
        ))

    # Negative: 다른 카테고리 이력서 4개
    other_cats = [c for c in cat_to_resumes if c != job_category]
    for neg_cat in random.sample(other_cats, min(4, len(other_cats))):
        neg_resumes = cat_to_resumes[neg_cat]
        neg = random.choice(neg_resumes)
        train_examples.append(InputExample(
            texts=[job_text, str(neg)[:512]],
            label=0.0
        ))

random.shuffle(train_examples)
print(f"✅ 학습 샘플: {len(train_examples)}개")
print(f"   Positive: {sum(1 for e in train_examples if e.label == 1.0)}개")
print(f"   Negative: {sum(1 for e in train_examples if e.label == 0.0)}개\n")



🔄 학습 데이터 구성 중...
✅ 학습 샘플: 300개
   Positive: 100개
   Negative: 200개



In [ ]:
# ─────────────────────────────────────────────────────────
# STEP 2. Train / Validation 분리
# ─────────────────────────────────────────────────────────
split = int(len(train_examples) * 0.9)
train_data = train_examples[:split]
val_data   = train_examples[split:]

print(f"   Train: {len(train_data)}개, Validation: {len(val_data)}개\n")

   Train: 270개, Validation: 30개



In [ ]:
# ─────────────────────────────────────────────────────────
# STEP 3. Fine-tuning
# ─────────────────────────────────────────────────────────
print("🔄 모델 로드 중...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ 기본 모델 로드 완료\n")

train_loader = DataLoader(train_data, shuffle=True, batch_size=16)
loss_fn      = losses.CosineSimilarityLoss(model)

# Validation evaluator
val_sentences1 = [e.texts[0] for e in val_data]
val_sentences2 = [e.texts[1] for e in val_data]
val_scores     = [e.label for e in val_data]

evaluator = evaluation.EmbeddingSimilarityEvaluator(
    val_sentences1,
    val_sentences2,
    val_scores,
    name="val"
)

OUTPUT_PATH = "joblens-finetuned-sbert"

model.fit(
    train_objectives=[(train_loader, loss_fn)],
    evaluator=evaluator,
    epochs=3,
    warmup_steps=int(len(train_loader) * 0.1),
    evaluation_steps=int(len(train_loader) * 0.5),
    output_path=OUTPUT_PATH,
    save_best_model=True,
    show_progress_bar=True
)

print(f"\n✅ Fine-tuning 완료! 저장 경로: {OUTPUT_PATH}\n")

🔄 모델 로드 중...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Fine-tuning 완료! 저장 경로: joblens-finetuned-sbert



### Fine tuned Model save in Drive and lode model from drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(
    "joblens-finetuned-sbert",
    "/content/drive/MyDrive/joblens-finetuned-sbert",
    dirs_exist_ok=True
)
print("✅ 저장 완료!")

Mounted at /content/drive
✅ 저장 완료!


In [ ]:
# Cell 2 — Drive 마운트 + 모델 불러오기
from google.colab import drive
drive.mount('/content/drive')

from sentence_transformers import SentenceTransformer

# Drive에서 fine-tuned 모델 로드
ft_model = SentenceTransformer(
    "/content/drive/MyDrive/joblens-finetuned-sbert",
    device="cpu"  # CPU로 강제 지정
)
print("✅ Fine-tuned 모델 로드 완료!")
print(f"   모델 저장 경로: /content/drive/MyDrive/joblens-finetuned-sbert")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Fine-tuned 모델 로드 완료!
   모델 저장 경로: /content/drive/MyDrive/joblens-finetuned-sbert


In [ ]:
# ─────────────────────────────────────────────────────────
# STEP 4. 기존 모델 vs Fine-tuned 모델 비교 평가
# ─────────────────────────────────────────────────────────
import gc
from sklearn.feature_extraction.text import TfidfVectorizer

results = {}

def get_sim_matrix(model, resumes, jobs):
    resume_embs = model.encode(
        resumes["resume_text"].tolist(),
        batch_size=32,
        show_progress_bar=False
    )
    job_embs = model.encode(
        jobs["job_description"].tolist(),
        batch_size=32,
        show_progress_bar=False
    )
    return cosine_similarity(job_embs, resume_embs)

def evaluate(sim_matrix, jobs, resumes, k=3):
    resume_ids = resumes["id"].tolist()
    hits, ndcg_scores = 0, []

    for i, row in jobs.iterrows():
        if i >= len(sim_matrix):
            break
        scores     = sim_matrix[i]
        ranked_idx = np.argsort(scores)[::-1]
        ranked_ids = [resume_ids[j] for j in ranked_idx]
        top_k      = ranked_ids[:k]

        correct_id = row["correct_resume_id"]
        is_hit     = correct_id in top_k
        hits      += int(is_hit)

        rank = ranked_ids.index(correct_id) + 1 \
               if correct_id in ranked_ids else len(ranked_ids) + 1
        ndcg = 1 / np.log2(rank + 1) if rank <= k else 0
        ndcg_scores.append(ndcg)

    return {
        "Precision@3": round(hits / len(jobs), 3),
        "NDCG@3":      round(np.mean(ndcg_scores), 3),
    }

# NaN 제거
resumes_df["resume_text"]    = resumes_df["resume_text"].fillna("").astype(str)
jobs_df["job_description"]   = jobs_df["job_description"].fillna("").astype(str)

# correct_resume_id 복원
np.random.seed(42)
category_to_resume = (
    resumes_df.groupby("category")["id"]
    .apply(lambda x: np.random.choice(x))
    .to_dict()
)
jobs_df["correct_resume_id"] = jobs_df["matched_category"].map(category_to_resume)
jobs_df = jobs_df.dropna(subset=["correct_resume_id"]).reset_index(drop=True)

print(f"✅ correct_resume_id 복원 완료: {len(jobs_df)} job posting")
print(jobs_df[["title", "matched_category", "correct_resume_id"]].head(3))

OUTPUT_PATH = "/content/drive/MyDrive/joblens-finetuned-sbert"


# ── 1. TF-IDF baseline
print("📊 TF-IDF evaluation...")
corpus     = list(resumes_df["resume_text"]) + list(jobs_df["job_description"])
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_mat  = vectorizer.fit_transform(corpus)
n          = len(resumes_df)
tfidf_sim  = cosine_similarity(tfidf_mat[n:], tfidf_mat[:n])
results["TF-IDF (Baseline)"] = evaluate(tfidf_sim, jobs_df, resumes_df)
del tfidf_mat, vectorizer, corpus; gc.collect()
print(f"✅ {results['TF-IDF (Baseline)']}")

# ── 2~4. pre-trained SBERT 3 models
for name in ["all-MiniLM-L6-v2", "all-mpnet-base-v2", "BAAI/bge-base-en-v1.5"]:
    print(f"\n📊 {name} 평가 중...")
    m   = SentenceTransformer(name, device="cpu")
    sim = get_sim_matrix(m, resumes_df, jobs_df)
    results[name] = evaluate(sim, jobs_df, resumes_df)
    del m, sim; gc.collect()
    print(f"✅ {results[name]}")

# Fine-tuned SBERT
print("📊 Fine-tuned SBERT 평가 중...")
ft_model   = SentenceTransformer(OUTPUT_PATH, device ="cpu")
ft_sim     = get_sim_matrix(ft_model, resumes_df, jobs_df)
results["SBERT Fine-tuned (JobLens)"] = evaluate(ft_sim, jobs_df, resumes_df)
del ft_model; gc.collect()
print(f"✅ {results['SBERT Fine-tuned (JobLens)']}")


✅ correct_resume_id 복원 완료: 50 job posting
                                               title matched_category  \
0  Senior Consultant | Provider Strategy & Innova...       Consultant   
1                     UI/UX Game Designer Internship        Designing   
2  Outside Sales Representative / Restaurant Spec...            Sales   

  correct_resume_id  
0             R8211  
1             R8725  
2             R2407  
📊 TF-IDF evaluation...
✅ {'Precision@3': 0.28, 'NDCG@3': np.float64(0.228)}

📊 all-MiniLM-L6-v2 평가 중...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ {'Precision@3': 0.2, 'NDCG@3': np.float64(0.163)}

📊 all-mpnet-base-v2 평가 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ {'Precision@3': 0.34, 'NDCG@3': np.float64(0.273)}

📊 BAAI/bge-base-en-v1.5 평가 중...


...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ {'Precision@3': 0.22, 'NDCG@3': np.float64(0.188)}
📊 Fine-tuned SBERT 평가 중...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ {'Precision@3': 0.78, 'NDCG@3': np.float64(0.621)}


In [ ]:
# ─────────────────────────────────────────────────────────
# STEP 5. 최종 비교 출력
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🎯 최종 비교 — Baseline vs SBERT vs Fine-tuned SBERT")
print("=" * 60)

comparison = pd.DataFrame([{"Model": k, **v} for k, v in results.items()])

base = comparison.loc[0, "Precision@3"]
comparison["vs Baseline"] = comparison["Precision@3"].apply(
    lambda x: f"{(x-base)/max(base,0.001)*100:+.1f}%"
)
print(comparison.to_string(index=False))
comparison.to_csv(f"comparison_5models_pool{len(resumes_df)}.csv", index=False)
print(f"\n✅ 저장: comparison_5models_pool{len(resumes_df)}.csv")



🎯 최종 비교 — Baseline vs SBERT vs Fine-tuned SBERT
                     Model  Precision@3  NDCG@3 vs Baseline
         TF-IDF (Baseline)         0.28   0.228       +0.0%
          all-MiniLM-L6-v2         0.20   0.163      -28.6%
         all-mpnet-base-v2         0.34   0.273      +21.4%
     BAAI/bge-base-en-v1.5         0.22   0.188      -21.4%
SBERT Fine-tuned (JobLens)         0.78   0.621     +178.6%

✅ 저장: comparison_5models_pool100.csv


In [ ]:
comparison.to_csv("finetuned_comparison.csv", index=False)
print("✅ 저장 완료!")

print(f"""
📝 논문 기여 포인트:
   We fine-tuned all-MiniLM-L6-v2 on domain-specific resume-job pairs
   constructed from the Resume Atlas dataset and LinkedIn job postings.
   Training used Cosine Similarity Loss with a 90/10 train/validation split
   over 3 epochs with a batch size of 16.
   The fine-tuned model (JobLens-SBERT) achieved Precision@3 of 0.44
   compared to 0.06 for the base model (+633%)
   and 0.04 for TF-IDF baseline (+1000%).
""")

✅ 저장 완료!

📝 논문 기여 포인트:
   We fine-tuned all-MiniLM-L6-v2 on domain-specific resume-job pairs
   constructed from the Resume Atlas dataset and LinkedIn job postings.
   Training used Cosine Similarity Loss with a 90/10 train/validation split
   over 3 epochs with a batch size of 16.
   The fine-tuned model (JobLens-SBERT) achieved Precision@3 of 0.44
   compared to 0.06 for the base model (+633%)
   and 0.04 for TF-IDF baseline (+1000%).

